# RZ-USFM project · Part 2 — Expert B: dual-stream deep model (lesion + peritumoral ring)

**Expert B** is the task-specific deep-learning expert: a **weight-shared dual-stream ConvNeXt-Tiny** (ImageNet-22k pretrained) that looks at
- **stream L** — the lesion (pixels inside the lesion mask), and
- **stream R** — the peritumoral ring (the same ring used for radiomics in Part 1),

concatenates both embeddings and predicts malignancy. It mirrors the intratumoral / peritumoral split of the radiomics, but learns its own features end-to-end with no radiomic constraint, which makes it an independent "second opinion" for the fusion.

**What this notebook does**
1. Builds a cache of square context crops (image + lesion mask + ring mask) once → fast training, no slow Drive reads per epoch.
2. Trains, for every outer fold, a **deep ensemble** of `N_SEEDS` dual-stream models (uncertainty = SD of member logits), plus a **single-stream baseline** (whole crop) for the ablation.
3. Uses a patient-level validation split *inside* each training fold for the decision threshold (Youden); the test fold is never used for any choice.
4. Saves out-of-fold predictions (`oof_expertB.csv`) and deep embeddings for Part 4.
5. Metrics with 95% CI, ROC / PR / calibration / confusion matrices, learning curves, t-SNE of embeddings (by label and by scanner), comparison with Expert A.
6. Explainability: **Grad-CAM** per stream, **SHAP** (image partition explainer) and **LIME** (superpixels) on TP / TN / FP / FN cases.

**Requires Part 1 first** (`master.csv`, `common.py`). Runtime: **T4 GPU**. Training is resumable: each (model, fold, seed) is saved as soon as it finishes.

## 1. Setup and paths

In [ ]:
!pip -q install timm shap lime scikit-image
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, time, json, warnings
warnings.filterwarnings('ignore')

PROJ    = '/content/drive/MyDrive/rsusfm_project'   # the project folder created by Part 1 (name kept so all parts find each other)
RUN_DIR = f'{PROJ}/results/expertB_runs'
FIG_DIR, RES_DIR = f'{PROJ}/figures/part2', f'{PROJ}/results'
for d in (RUN_DIR, FIG_DIR): os.makedirs(d, exist_ok=True)
assert os.path.exists(f'{PROJ}/master.csv') and os.path.exists(f'{PROJ}/common.py'), 'Run Part 1 first (master.csv, common.py)'

BACKBONE   = 'convnext_tiny.fb_in22k_ft_in1k'
PRETRAINED = True
IMG, CACHE_SIZE = 224, 288      # network input; cached crop size (room for augmentation)
EPOCHS, BS, LR_BACKBONE, LR_HEAD, WD = 20, 32, 1e-4, 1e-3, 0.05
N_SEEDS    = 3                  # deep-ensemble members for the dual-stream model (Expert B)
RUN_SINGLE_STREAM_BASELINE = True   # 1 seed per fold, for the ablation "dual vs single stream"
SEED = 42

In [ ]:
sys.path.insert(0, PROJ)
import importlib, common; importlib.reload(common)
from common import *
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F, timm
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, torch.cuda.get_device_name(0) if device == 'cuda' else '')
master = pd.read_csv(f'{PROJ}/master.csv')
print(master.shape, '| folds:', sorted(master.fold.unique()))

## 2. Crop cache: square context box around lesion + ring
- The ring is computed exactly as in Part 1 (20% of the lesion's equivalent diameter, min 5 px), on the full image.
- Box = bounding box of lesion ∪ ring, enlarged by 25% and made square (zero padding at the image border), resized to 288×288.
- Saved once to Drive (`expertB_crops_288.npz`) and reloaded afterwards.

In [ ]:
from PIL import Image
from scipy import ndimage as ndi
from joblib import Parallel, delayed

def ring_from(lesion, frac=0.20, min_px=5):          # identical to Part 1
    diam = 2 * np.sqrt(lesion.sum() / np.pi)
    w = max(min_px, int(round(frac * diam)))
    dist = ndi.distance_transform_edt(~lesion)
    return (dist > 0) & (dist <= w)

def make_crop(img_path, mask_path, out=CACHE_SIZE, margin=0.25):
    img = np.array(Image.open(img_path).convert('L'), dtype=np.uint8)
    les = np.array(Image.open(mask_path).convert('L')) > 127
    if les.shape != img.shape:
        les = np.array(Image.fromarray(les.astype(np.uint8) * 255).resize(img.shape[::-1], Image.NEAREST)) > 127
    ring = ring_from(les)
    ys, xs = np.where(les | ring)
    cy, cx = (ys.min() + ys.max()) / 2, (xs.min() + xs.max()) / 2
    half = max(ys.max() - ys.min(), xs.max() - xs.min()) / 2 * (1 + margin) + 2
    y0, x0, s = int(round(cy - half)), int(round(cx - half)), int(round(2 * half))
    def cut(a):
        pad = np.zeros((s, s), a.dtype)
        sy0, sx0 = max(0, y0), max(0, x0); sy1, sx1 = min(a.shape[0], y0 + s), min(a.shape[1], x0 + s)
        pad[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = a[sy0:sy1, sx0:sx1]
        return pad
    im = np.array(Image.fromarray(cut(img)).resize((out, out), Image.BILINEAR))
    ml = np.array(Image.fromarray(cut(les.astype(np.uint8))).resize((out, out), Image.NEAREST))
    mr = np.array(Image.fromarray(cut(ring.astype(np.uint8))).resize((out, out), Image.NEAREST))
    return im, ml, mr

CACHE = f'{PROJ}/results/expertB_crops_{CACHE_SIZE}.npz'
if os.path.exists(CACHE):
    z = np.load(CACHE, allow_pickle=True)
    assert list(z['ids']) == list(master.id), 'cache does not match master.csv order - delete it and rebuild'
    CROP_IMG, CROP_LES, CROP_RING = z['img'], z['les'], z['ring']
else:
    t0 = time.time()
    res = Parallel(n_jobs=-1)(delayed(make_crop)(r.img_path, r.mask_path) for r in master.itertuples())
    CROP_IMG, CROP_LES, CROP_RING = (np.stack(a) for a in zip(*res))
    np.savez_compressed(CACHE, img=CROP_IMG, les=CROP_LES, ring=CROP_RING, ids=master.id.values)
    print(f'cache built in {(time.time() - t0) / 60:.1f} min')
print('crops:', CROP_IMG.shape, '| mean lesion fraction:', round(CROP_LES.mean(), 3), '| ring fraction:', round(CROP_RING.mean(), 3))

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for k, i in enumerate(np.random.default_rng(SEED).choice(len(master), 6, replace=False)):
    axes[0, k].imshow(CROP_IMG[i], cmap='gray'); axes[0, k].set_title(['benign', 'malignant'][master.label[i]], fontsize=9)
    ov = np.stack([CROP_IMG[i]] * 3, -1).astype(float) / 255
    ov[CROP_LES[i] > 0] = 0.5 * ov[CROP_LES[i] > 0] + 0.5 * np.array([1, .2, .2])
    ov[CROP_RING[i] > 0] = 0.5 * ov[CROP_RING[i] > 0] + 0.5 * np.array([.2, .6, 1])
    axes[1, k].imshow(ov)
for a in axes.ravel(): a.axis('off')
fig.suptitle('Context crops (top) with lesion (red) and peritumoral ring (blue) — the two input streams')
fig.savefig(f'{FIG_DIR}/input_streams_examples.png', bbox_inches='tight'); plt.show()

## 3. Dataset, augmentation and model
- **Augmentation** (training only): the same random flip + affine is applied to image and both masks; brightness/contrast jitter on the image only.
- Inputs are ImageNet-normalised; outside a mask the value is 0, i.e. the ImageNet mean.
- **Dual-stream model:** one ConvNeXt-Tiny with **shared weights** processes L and R (halves the parameters → less overfitting), embeddings [f(L); f(R)] → LayerNorm → dropout → linear → logit.
- **Single-stream baseline:** the same backbone on the whole context crop.

In [ ]:
import torchvision.transforms.v2 as T2
from torchvision import tv_tensors
from torch.utils.data import Dataset, DataLoader
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

GEOM = T2.Compose([T2.RandomHorizontalFlip(), T2.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1))])
PHOTO = T2.ColorJitter(brightness=0.2, contrast=0.2)

class CropDS(Dataset):
    def __init__(self, idx, train): self.idx, self.train = np.asarray(idx), train
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]
        im = tv_tensors.Image(torch.from_numpy(CROP_IMG[i]).float().div(255)[None].repeat(3, 1, 1))
        ml = tv_tensors.Mask(torch.from_numpy(CROP_LES[i])[None]); mr = tv_tensors.Mask(torch.from_numpy(CROP_RING[i])[None])
        if self.train:
            im, ml, mr = GEOM(im, ml, mr); im = PHOTO(im)
        im = F.interpolate(im[None], size=(IMG, IMG), mode='bilinear', align_corners=False)[0]
        ml = F.interpolate(ml[None].float(), size=(IMG, IMG), mode='nearest')[0]
        mr = F.interpolate(mr[None].float(), size=(IMG, IMG), mode='nearest')[0]
        x = (im - MEAN) / STD
        return x, ml, mr, torch.tensor(float(master.label[i]))

class DeepExpert(nn.Module):
    def __init__(self, dual=True, backbone=BACKBONE, pretrained=PRETRAINED):
        super().__init__()
        self.dual = dual
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0, drop_path_rate=0.1)
        d = self.backbone.num_features * (2 if dual else 1)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, 1))
    def embed(self, x, ml, mr):
        if not self.dual: return self.backbone(x)
        f = self.backbone(torch.cat([x * ml, x * mr]))            # both streams in one pass, shared weights
        return torch.cat(f.chunk(2), dim=1)                        # [f(L) ; f(R)]
    def forward(self, x, ml, mr):
        e = self.embed(x, ml, mr)
        return self.head(e).squeeze(1), e

print('trainable parameters (dual):', sum(p.numel() for p in DeepExpert(pretrained=False).parameters()) / 1e6, 'M')

## 4. Training loop (one model = one variant × fold × seed)
AdamW (backbone lr 1e-4, head 1e-3), 1-epoch warm-up + cosine decay, class-weighted BCE, mixed precision, fixed number of epochs (no early stopping on any test data). Inference uses horizontal-flip test-time augmentation.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
y_all, g_all = master.label.values, master.patient.astype(str).values

def val_split(f):
    tr = np.where(master.fold != f)[0]
    sgk = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED + f)
    a, b = next(sgk.split(tr, y_all[tr], g_all[tr]))
    return tr[a], tr[b]                                            # fit set, validation set (patient-disjoint)

@torch.no_grad()
def predict(model, idx, tta=True):
    model.eval(); dl = DataLoader(CropDS(idx, False), batch_size=64, num_workers=2)
    L, E = [], []
    for x, ml, mr, _ in dl:
        x, ml, mr = x.to(device), ml.to(device), mr.to(device)
        with torch.autocast('cuda', dtype=torch.float16, enabled=device == 'cuda'):
            lo, e = model(x, ml, mr)
            if tta:
                lo2, _ = model(x.flip(-1), ml.flip(-1), mr.flip(-1)); lo = (lo + lo2) / 2
        L.append(lo.float().cpu()); E.append(e.float().cpu())
    return torch.cat(L).numpy(), torch.cat(E).numpy()

def train_one(variant, f, seed, save_weights=False):
    path = f'{RUN_DIR}/{variant}_f{f}_s{seed}.npz'
    if os.path.exists(path): return dict(np.load(path))
    torch.manual_seed(seed); np.random.seed(seed)
    fit_idx, val_idx = val_split(f); test_idx = np.where(master.fold == f)[0]
    model = DeepExpert(dual=(variant == 'dual')).to(device)
    dl = DataLoader(CropDS(fit_idx, True), batch_size=BS, shuffle=True, num_workers=2, drop_last=True)
    head = list(model.head.parameters()); back = list(model.backbone.parameters())
    opt = torch.optim.AdamW([{'params': back, 'lr': LR_BACKBONE}, {'params': head, 'lr': LR_HEAD}], weight_decay=WD)
    steps = EPOCHS * len(dl); warm = len(dl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warm) * 0.5 * (1 + np.cos(np.pi * min(1, s / steps))))
    pw = torch.tensor((1 - y_all[fit_idx].mean()) / y_all[fit_idx].mean(), device=device)
    scaler = torch.amp.GradScaler(enabled=device == 'cuda')
    curve = []
    for ep in range(EPOCHS):
        model.train(); t0, tot = time.time(), 0.0
        for x, ml, mr, y in dl:
            x, ml, mr, y = x.to(device), ml.to(device), mr.to(device), y.to(device)
            with torch.autocast('cuda', dtype=torch.float16, enabled=device == 'cuda'):
                lo, _ = model(x, ml, mr)
                loss = F.binary_cross_entropy_with_logits(lo.float(), y, pos_weight=pw)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item()
        lv, _ = predict(model, val_idx, tta=False)
        auc_v = roc_auc_score(y_all[val_idx], lv)
        curve.append((tot / len(dl), auc_v))
        print(f'  [{variant} f{f} s{seed}] epoch {ep + 1:2d}/{EPOCHS} loss {tot / len(dl):.4f} val AUC {auc_v:.3f} ({time.time() - t0:.0f}s)')
    lv, _ = predict(model, val_idx); lt, et = predict(model, test_idx)
    out = dict(val_idx=val_idx, val_logit=lv, test_idx=test_idx, test_logit=lt, curve=np.array(curve))
    if seed == SEED: out['test_emb'] = et.astype(np.float16)
    np.savez(path, **out)
    if save_weights: torch.save(model.state_dict(), f'{RUN_DIR}/{variant}_f{f}_s{seed}.pt')
    del model; torch.cuda.empty_cache()
    return out

## 5. Run all folds
Dual-stream: `N_SEEDS` seeds per fold (Expert B = their ensemble). Single-stream baseline: 1 seed per fold.
The fold-0 / seed-0 dual model weights are kept for the explainability section. Re-run this cell after a disconnection: finished models are skipped.

In [ ]:
SEEDS = [SEED + s for s in range(N_SEEDS)]
plan = [('dual', f, s) for f in sorted(master.fold.unique()) for s in SEEDS]
if RUN_SINGLE_STREAM_BASELINE: plan += [('single', f, SEED) for f in sorted(master.fold.unique())]
T0 = time.time()
for k, (v, f, s) in enumerate(plan):
    t = time.time(); train_one(v, f, s, save_weights=(v == 'dual' and f == 0 and s == SEED))
    print(f'>>> {k + 1}/{len(plan)} done: {v} fold {f} seed {s} ({(time.time() - t) / 60:.1f} min, total {(time.time() - T0) / 60:.0f} min)')

## 6. Out-of-fold predictions, uncertainty and thresholds

In [ ]:
def assemble(variant, seeds):
    o = master[['id', 'patient', 'fold', 'label']].copy()
    o[['prob', 'logit_mean', 'logit_sd', 'thr']] = np.nan
    emb = None
    for f in sorted(master.fold.unique()):
        runs = [dict(np.load(f'{RUN_DIR}/{variant}_f{f}_s{s}.npz')) for s in seeds]
        te, va = runs[0]['test_idx'], runs[0]['val_idx']
        Lt = np.stack([r['test_logit'] for r in runs]); Lv = np.stack([r['val_logit'] for r in runs]).mean(0)
        thr = youden_threshold(y_all[va], 1 / (1 + np.exp(-Lv)))          # chosen on validation (training data)
        mu = Lt.mean(0)
        o.loc[te, 'logit_mean'] = mu; o.loc[te, 'prob'] = 1 / (1 + np.exp(-mu))
        o.loc[te, 'logit_sd'] = Lt.std(0) if len(seeds) > 1 else np.nan
        o.loc[te, 'thr'] = thr
        if 'test_emb' in runs[0]:
            if emb is None: emb = np.full((len(master), runs[0]['test_emb'].shape[1]), np.nan, np.float32)
            emb[te] = runs[0]['test_emb']
    return o, emb

OOF = {}
OOF['Deep-DualStream-Ensemble'], EMB_B = assemble('dual', SEEDS)
OOF['Deep-DualStream-1seed'], _ = assemble('dual', [SEED])
if RUN_SINGLE_STREAM_BASELINE: OOF['Deep-SingleStream'], _ = assemble('single', [SEED])

save_oof(OOF['Deep-DualStream-Ensemble'], 'expertB', RES_DIR)
for n, o in OOF.items(): save_oof(o, n, RES_DIR)
np.save(f'{RES_DIR}/expertB_emb_oof.npy', EMB_B)          # row order = master.csv (for the early-fusion baseline, Part 4)
print('saved oof_expertB.csv and expertB_emb_oof.npy')

## 7. Results: metrics (95% CI) and figures

In [ ]:
rows = [evaluate_oof(o, n, RES_DIR) for n, o in OOF.items()]
display(summary_table(rows))
pd.DataFrame(rows).to_csv(f'{RES_DIR}/part2_metrics_full.csv', index=False)
plot_roc(OOF, f'{FIG_DIR}/roc_expertB.png', 'ROC — deep models (out-of-fold)')
plot_pr(OOF, f'{FIG_DIR}/pr_expertB.png')
plot_calibration(OOF, f'{FIG_DIR}/calibration_expertB.png')
for n, o in OOF.items(): plot_confusion(o, n, f'{FIG_DIR}/cm_{n}.png')
plot_metric_bars(rows, f'{FIG_DIR}/metrics_bars_expertB.png')

# Expert A vs Expert B (full comparison of all experts in Part 4)
if os.path.exists(f'{RES_DIR}/oof_expertA.csv'):
    A = pd.read_csv(f'{RES_DIR}/oof_expertA.csv')
    AB = {'Expert A (radiomics)': A, 'Expert B (deep, dual-stream)': OOF['Deep-DualStream-Ensemble']}
    plot_roc(AB, f'{FIG_DIR}/roc_expertA_vs_B.png', 'Expert A vs Expert B')
    rows_ab = [evaluate_oof(A, 'Expert A (radiomics)', RES_DIR), {**rows[0], 'model': 'Expert B (deep, dual-stream)'}]
    display(summary_table(rows_ab))
    plot_metric_bars(rows_ab, f'{FIG_DIR}/metrics_bars_expertA_vs_B.png')

## 8. Learning curves, uncertainty and embedding map

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for f in sorted(master.fold.unique()):
    c = np.load(f'{RUN_DIR}/dual_f{f}_s{SEED}.npz')['curve']
    axes[0].plot(range(1, len(c) + 1), c[:, 0], label=f'fold {f}'); axes[1].plot(range(1, len(c) + 1), c[:, 1], label=f'fold {f}')
axes[0].set(xlabel='Epoch', ylabel='Training loss', title='Training loss (dual-stream, seed 0)')
axes[1].set(xlabel='Epoch', ylabel='Validation AUC', title='Validation AUC (inner split)'); axes[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(f'{FIG_DIR}/learning_curves.png', bbox_inches='tight'); plt.show()

# Does the ensemble uncertainty flag errors?  (needed for the Bayesian fusion)
o = OOF['Deep-DualStream-Ensemble']
if o.logit_sd.notna().all():
    err = ((o.prob >= o.thr).astype(int) != o.label).astype(int)
    print(f'AUROC of ensemble SD for detecting errors: {roc_auc_score(err, o.logit_sd):.3f}  (0.5 = no information)')
    fig, ax = plt.subplots(figsize=(4.5, 3.5))
    ax.boxplot([o.logit_sd[err == 0], o.logit_sd[err == 1]])
    ax.set_xticks([1, 2], ['correct', 'error'])
    ax.set(ylabel='Ensemble logit SD (uncertainty)', title='Expert B uncertainty vs errors')
    fig.savefig(f'{FIG_DIR}/uncertainty_vs_errors_expertB.png', bbox_inches='tight'); plt.show()

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
ok = ~np.isnan(EMB_B).any(1)
Z = TSNE(2, perplexity=30, random_state=SEED, init='pca').fit_transform(PCA(50, random_state=SEED).fit_transform(EMB_B[ok]))
cols = [('label', ['benign', 'malignant'])] + ([('device', None)] if 'device' in master else [])
fig, axes = plt.subplots(1, len(cols), figsize=(5.5 * len(cols), 4.8))
axes = np.atleast_1d(axes)
for ax, (col, names) in zip(axes, cols):
    v = master.loc[ok, col].values
    for u in np.unique(v):
        ax.scatter(*Z[v == u].T, s=6, alpha=.7, label=names[int(u)] if names else str(u))
    ax.set(title=f't-SNE of Expert B embeddings — coloured by {col}', xticks=[], yticks=[]); ax.legend(fontsize=8, markerscale=2)
fig.savefig(f'{FIG_DIR}/tsne_expertB.png', bbox_inches='tight'); plt.show()
print('If points cluster by scanner more than by label, the deep features carry scanner shortcuts - '
      'exactly what the radiomic regularisation of RZ-USFM (Part 3) is meant to reduce.')

## 9. Explainability on fold-0 test cases (TP, TN, FP, FN)
- **Grad-CAM** for each stream (lesion and ring), w.r.t. the malignancy logit.
- **SHAP**: partition explainer with a blur masker on the crop (masks kept fixed) — which image regions push the prediction.
- **LIME**: SLIC superpixels restricted to lesion + ring (the only pixels the dual-stream model sees); green = regions supporting malignancy, red = against.

In [ ]:
model = DeepExpert(dual=True).to(device)
model.load_state_dict(torch.load(f'{RUN_DIR}/dual_f0_s{SEED}.pt', map_location=device)); model.eval()
o0 = OOF['Deep-DualStream-1seed']; te0 = np.where(master.fold == 0)[0]
pred = (o0.prob >= o0.thr).astype(int).values; lab = master.label.values
picks = {}
for name, cond in {'TP': (lab == 1) & (pred == 1), 'TN': (lab == 0) & (pred == 0),
                   'FP': (lab == 0) & (pred == 1), 'FN': (lab == 1) & (pred == 0)}.items():
    c = [i for i in te0 if cond[i]]
    if c: picks[name] = c[int(np.argmax(np.abs(o0.logit_mean.values[c])))]       # most confident case of each type
print('cases:', {k: master.id[v] for k, v in picks.items()})

def tensors(i):
    x, ml, mr, _ = CropDS([i], False)[0]
    return x[None].to(device), ml[None].to(device), mr[None].to(device)
def to_rgb(x): return ((x[0].cpu() * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy())

acts = {}
h1 = model.backbone.stages[-1].register_forward_hook(lambda m, i, o: acts.__setitem__('a', o))
def gradcam(i):
    x, ml, mr = tensors(i)
    with torch.enable_grad():
        model.zero_grad(); lo, _ = model(x, ml, mr)
        a = acts['a']; g = torch.autograd.grad(lo.sum(), a)[0]         # (2, C, h, w): stream L, stream R
        cam = F.relu((g.mean((2, 3), keepdim=True) * a).sum(1))
    cam = F.interpolate(cam[:, None], size=(IMG, IMG), mode='bilinear')[:, 0].detach().cpu().numpy()
    return [c / (c.max() + 1e-8) for c in cam], torch.sigmoid(lo).item()

fig, axes = plt.subplots(len(picks), 3, figsize=(9, 3 * len(picks)))
axes = np.atleast_2d(axes)
for r, (name, i) in enumerate(picks.items()):
    (cl, cr), p = gradcam(i); x, ml, mr = tensors(i); rgb = to_rgb(x)
    axes[r, 0].imshow(rgb); axes[r, 0].contour(ml[0, 0].cpu(), [0.5], colors='r', linewidths=1)
    axes[r, 0].contour(mr[0, 0].cpu(), [0.5], colors='c', linewidths=.8)
    axes[r, 0].set_title(f'{name}: {["benign", "malignant"][lab[i]]}, p = {p:.2f}', fontsize=9)
    axes[r, 1].imshow(rgb); axes[r, 1].imshow(cl, cmap='jet', alpha=.45); axes[r, 1].set_title('Grad-CAM, lesion stream', fontsize=9)
    axes[r, 2].imshow(rgb); axes[r, 2].imshow(cr, cmap='jet', alpha=.45); axes[r, 2].set_title('Grad-CAM, ring stream', fontsize=9)
for a in axes.ravel(): a.axis('off')
fig.tight_layout(); fig.savefig(f'{FIG_DIR}/gradcam_expertB.png', bbox_inches='tight'); plt.show()
h1.remove()

In [ ]:
import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries, slic

def make_f(i):
    _, ml, mr = tensors(i)
    @torch.no_grad()
    def f(batch):                                    # batch: (N, H, W, 3) RGB in [0, 1]
        out = []
        for s in range(0, len(batch), 32):
            b = torch.tensor(np.asarray(batch[s:s + 32]), dtype=torch.float32, device=device).permute(0, 3, 1, 2)
            b = (b - MEAN.to(device)) / STD.to(device)
            n = len(b)
            with torch.autocast('cuda', dtype=torch.float16, enabled=device == 'cuda'):
                lo, _ = model(b, ml.expand(n, -1, -1, -1), mr.expand(n, -1, -1, -1))
            out.append(torch.sigmoid(lo.float()).cpu().numpy())
        p = np.concatenate(out); return np.stack([1 - p, p], 1)
    return f

masker = shap.maskers.Image('blur(16,16)', (IMG, IMG, 3))
lime_exp = lime_image.LimeImageExplainer(random_state=SEED)
def seg_fn_for(support):        # superpixels only inside lesion + ring (the only pixels the model sees)
    return lambda img: slic(img, n_segments=40, compactness=10, start_label=1, mask=support, channel_axis=-1)
fig, axes = plt.subplots(len(picks), 3, figsize=(9, 3 * len(picks)))
axes = np.atleast_2d(axes)
for r, (name, i) in enumerate(picks.items()):
    x, ml, mr = tensors(i); rgb = to_rgb(x); f = make_f(i)
    support = ((ml + mr)[0, 0] > 0).cpu().numpy()
    sv = shap.Explainer(lambda b: f(b)[:, 1:], masker)(rgb[None], max_evals=1000, batch_size=50)
    s = sv.values[0, ..., 0].sum(-1); lim = np.abs(s).max() + 1e-8
    e = lime_exp.explain_instance(rgb, f, labels=(1,), hide_color=0, num_samples=500, segmentation_fn=seg_fn_for(support))
    tmp, msk = e.get_image_and_mask(1, positive_only=False, num_features=6, hide_rest=False)
    axes[r, 0].imshow(rgb); axes[r, 0].contour(ml[0, 0].cpu(), [0.5], colors='r', linewidths=1)
    axes[r, 0].set_title(f'{name}: {["benign", "malignant"][lab[i]]}, p = {f(rgb[None])[0, 1]:.2f}', fontsize=9)
    axes[r, 1].imshow(rgb, alpha=.6); axes[r, 1].imshow(s, cmap='bwr', vmin=-lim, vmax=lim, alpha=.6)
    axes[r, 1].contour(ml[0, 0].cpu(), [0.5], colors='k', linewidths=.8); axes[r, 1].contour(mr[0, 0].cpu(), [0.5], colors='k', linewidths=.5, linestyles='--')
    axes[r, 1].set_title('SHAP (red → malignant)', fontsize=9)
    axes[r, 2].imshow(mark_boundaries(tmp, msk)); axes[r, 2].set_title('LIME superpixels', fontsize=9)
    ov = np.zeros((*msk.shape, 4)); ov[msk == 1] = [0, 1, 0, .35]; ov[msk == -1] = [1, 0, 0, .35]; axes[r, 2].imshow(ov)
for a in axes.ravel(): a.axis('off')
fig.suptitle('Expert B explanations: SHAP and LIME (green = supports malignancy, red = against)', fontsize=10)
fig.tight_layout(); fig.savefig(f'{FIG_DIR}/shap_lime_expertB.png', bbox_inches='tight'); plt.show()

## 10. Outputs of Part 2
| File | Used by |
|---|---|
| `results/oof_expertB.csv` | **Part 4 fusion** (prob, logit mean, logit SD = ensemble uncertainty, threshold) |
| `results/oof_Deep-*.csv`, `metrics_Deep-*.json`, `part2_metrics_full.csv` | Part 4 comparison tables, ablation (dual vs single stream, ensemble vs single seed) |
| `results/expertB_emb_oof.npy` | Part 4 early-fusion baseline (deep + radiomic features) |
| `results/expertB_crops_288.npz` | **reused by Part 3** (same crops and ring for RZ-USFM) |
| `results/expertB_runs/*.npz` | per-model predictions (resumability) |
| `figures/part2/*.png` | paper figures (300 dpi) |